# Chapter 10: Introduction to Artificial Neural Networks with Keras

## From Biological to Artificial Neurons

The chapter begins by establishing the inspiration behind Artificial Neural Networks (ANNs). Just as planes were inspired by birds but do not flap their wings, ANNs are inspired by biological neurons but have evolved into their own distinct architectures.


### The Perceptron

The Perceptron, invented by Frank Rosenblatt in 1957, is one of the simplest ANN architectures2. It is based on the Threshold Logic Unit (TLU).
- Architecture: Input connections have weights. The TLU computes a weighted sum of its inputs ($z = \mathbf{x}^T \mathbf{w}$) and applies a step function (like Heaviside or Sign) to output a result.
- Training: It uses Hebb’s rule ("Cells that fire together, wire together"). The connection weight between two neurons is increased whenever they have the same output. Rosenblatt formalized this into a rule that reinforces connections that help reduce error.
- Limitation: Perceptrons are linear classifiers and cannot solve simple non-linear problems like the Exclusive OR (XOR)5.

<p align="left"><img src="../fig/figure10.1.png" width="45%"></p>

### The Multilayer Perceptron (MLP) and Backpropagation

To solve non-linear problems (like XOR), we stack Perceptrons to create a Multilayer Perceptron (MLP). An MLP consists of:
1. Input Layer: Passthrough neurons.
2. Hidden Layers: One or more layers of TLUs.
3. Output Layer: Final layer of TLUs.

Backpropagation is the algorithm used to train MLPs. It relies on Gradient Descent and automatic differentiation (autodiff).

- Process:
1. Forward Pass: Data flows through the network to compute predictions.
2. Error Calculation: The loss function measures the difference between prediction and target.
3. Reverse Pass: The chain rule is used to propagate the error backward, measuring how much each weight contributed to the error.
4. Weight Update: Gradient Descent tweaks weights to reduce error.

Crucial Change: For backpropagation to work, the step function was replaced with differentiable activation functions, such as the Sigmoid (Logistic) function, Hyperbolic Tangent (tanh), or the Rectified Linear Unit (ReLU)10101010. ReLU ($ReLU(z) = max(0, z)$) is the most common default today because it is fast and reduces vanishing gradient issues11.

<p align="left"><img src="../fig/figure10.7.png" width="45%"></p>

## Implementing MLPs with Keras

Keras is a high-level Deep Learning API. We use tf.keras, which is the implementation bundled with TensorFlow.

<p align="left"><img src="../fig/figure10.10.png" width="45%"></p>

### Building an Image Classifier (Sequential API)
We will use the Fashion MNIST dataset (70,000 grayscale images of fashion items, 28x28 pixels, 10 classes).

Step 1: Load and Preprocess Data We load the data and scale pixel intensities to the 0-1 range (dividing by 255.0) to help Gradient Descent. We also split the training set to create a validation set.

In [2]:
import tensorflow as tf
from tensorflow import keras

# Load the dataset
fashion_mnist = keras.datasets.fashion_mnist
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist.load_data()

# Create validation set and scale pixel intensities
X_valid, X_train = X_train_full[:5000] / 255.0, X_train_full[5000:] / 255.0
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]

# Define class names
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Step 2: Build the Model We use the Sequential API, which is a simple stack of layers.
- Flatten: Converts the 2D image (28x28) into a 1D array.
- Dense: A fully connected layer. We use ReLU for hidden layers and Softmax for the output layer (because classes are exclusive).

In [3]:
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.Dense(300, activation="relu"),
    keras.layers.Dense(100, activation="relu"),
    keras.layers.Dense(10, activation="softmax")
])

model.summary() # Inspect the model architecture

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 300)            │       235,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 100)            │        30,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         1,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 266,610 (1.02 MB)

 Trainable params: 266,610 (1.02 MB)

 Non-trainable params: 0 (0.00 B)

Step 3: Compile the Model We specify the loss function (sparse_categorical_crossentropy because we have sparse labels, not one-hot vectors), the optimizer (sgd for Stochastic Gradient Descent), and metrics (accuracy).

In [4]:
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

Step 4: Train and Evaluate We call fit() to train the model. Keras displays the loss and accuracy for both training and validation sets at each epoch.

In [7]:
import numpy as np
# Train the model
history = model.fit(X_train, y_train, epochs=30,
                    validation_data=(X_valid, y_valid))

# Evaluate on test set
model.evaluate(X_test, y_test)

# Make predictions
X_new = X_test[:3]
y_proba = model.predict(X_new)
y_pred = np.argmax(model.predict(X_new), axis=-1)
print(y_pred)

Epoch 1/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9526 - loss: 0.1374 - val_accuracy: 0.8920 - val_loss: 0.3145
Epoch 2/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9527 - loss: 0.1355 - val_accuracy: 0.8892 - val_loss: 0.3322
Epoch 3/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9521 - loss: 0.1351 - val_accuracy: 0.8980 - val_loss: 0.3017
Epoch 4/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9517 - loss: 0.1366 - val_accuracy: 0.8924 - val_loss: 0.3089
Epoch 5/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9544 - loss: 0.1282 - val_accuracy: 0.8970 - val_loss: 0.3075
Epoch 6/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9575 - loss: 0.1255 - val_accuracy: 0.8986 - val_loss: 0.3054
Epoch 7/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9558 - loss: 0.1267 - val_accuracy: 0.8910 - val_loss: 0.3191
Epoch 8/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9566 - loss: 0.1224 - 

## Regression MLPs
MLPs are also used for regression. The main differences are:
1. Output Layer: Usually one neuron (for single value prediction) with no activation function (or ReLU if output must be positive).
2. Loss Function: Mean Squared Error (MSE).

Here is an example using the California Housing dataset:

In [8]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load and process data
housing = fetch_california_housing()
X_train_full, X_test, y_train_full, y_test = train_test_split(housing.data, housing.target)
X_train, X_valid, y_train, y_valid = train_test_split(X_train_full, y_train_full)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)

# Build Regression MLP
model = keras.models.Sequential([
    keras.layers.Dense(30, activation="relu", input_shape=X_train.shape[1:]),
    keras.layers.Dense(1)
])

model.compile(loss="mean_squared_error", optimizer="sgd")
history = model.fit(X_train, y_train, epochs=20, validation_data=(X_valid, y_valid))

Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


363/363 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 1.4669 - val_loss: 0.5179
Epoch 2/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5445 - val_loss: 0.4691
Epoch 3/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4547 - val_loss: 0.4509
Epoch 4/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4275 - val_loss: 0.4307
Epoch 5/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4248 - val_loss: 0.4219
Epoch 6/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4340 - val_loss: 0.4099
Epoch 7/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4061 - val_loss: 0.4008
Epoch 8/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3859 - val_loss: 0.3910
Epoch 9/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4019 - val_loss: 0.3874
Epoch 10/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3802 - val_loss: 0.3908
Epoch 11/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3796 - val_loss: 0.3808
Epoch 12/20
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.

## Complex Models: Functional and Subclassing APIs
The Functional API
The Sequential API is limited to simple stacks of layers. For complex topologies (e.g., Wide & Deep networks where inputs connect directly to the output to preserve simple rules), we use the Functional API.

In this example, we manipulate layers as functions. We pass inputs through a deep path (hidden layers) and concatenate them with the original inputs before the output layer.

In [9]:
# Wide & Deep Neural Network
input_ = keras.layers.Input(shape=X_train.shape[1:])
hidden1 = keras.layers.Dense(30, activation="relu")(input_)
hidden2 = keras.layers.Dense(30, activation="relu")(hidden1)
concat = keras.layers.Concatenate()([input_, hidden2])
output = keras.layers.Dense(1)(concat)

model = keras.Model(inputs=[input_], outputs=[output])

This API also supports Multiple Inputs (e.g., sending different features to different paths) and Multiple Outputs (e.g., for regularization or multitask learning).

### The Subclassing API
For dynamic behaviors (loops, conditional branching), you can subclass the Model class. You define layers in __init__ and the computation logic in call().

In [10]:
class WideAndDeepModel(keras.Model):
    def __init__(self, units=30, activation="relu", **kwargs):
        super().__init__(**kwargs)
        self.hidden1 = keras.layers.Dense(units, activation=activation)
        self.hidden2 = keras.layers.Dense(units, activation=activation)
        self.main_output = keras.layers.Dense(1)
        self.aux_output = keras.layers.Dense(1)

    def call(self, inputs):
        input_A, input_B = inputs
        hidden1 = self.hidden1(input_B)
        hidden2 = self.hidden2(hidden1)
        concat = keras.layers.concatenate([input_A, hidden2])
        main_output = self.main_output(concat)
        aux_output = self.aux_output(hidden2)
        return main_output, aux_output

model = WideAndDeepModel()

## Fine-Tuning Hyperparameters

Neural networks have many hyperparameters (layers, neurons, learning rate, batch size). Instead of manual guessing, we can use RandomizedSearchCV from Scikit-Learn by wrapping the Keras model in a KerasRegressor (or KerasClassifier).

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from scipy.stats import reciprocal
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_california_housing
from scikeras.wrappers import KerasRegressor
import sklearn

# Cek versi untuk memastikan downgrade berhasil (Harus 1.5.2)
print(f"Versi Scikit-Learn saat ini: {sklearn.__version__}")

# --- RELOAD DATA  ---
housing = fetch_california_housing()
X_train_full, X_test, y_train_full, y_test = train_test_split(housing.data, housing.target, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_train_full, y_train_full, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)

# --- BUILD MODEL ---
def build_model(n_hidden=1, n_neurons=30, learning_rate=3e-3, input_shape=[8]):
    model = keras.models.Sequential()
    model.add(keras.layers.InputLayer(input_shape=input_shape))
    for layer in range(n_hidden):
        model.add(keras.layers.Dense(n_neurons, activation="relu"))
    model.add(keras.layers.Dense(1))

    # Gunakan parameter learning_rate (bukan lr)
    optimizer = keras.optimizers.SGD(learning_rate=learning_rate)
    model.compile(loss="mse", optimizer=optimizer)
    return model

# --- WRAPPER ---
keras_reg = KerasRegressor(model=build_model, verbose=0)

# --- PARAMETER SEARCH ---
param_distribs = {
    "model__n_hidden": [0, 1, 2, 3],
    "model__n_neurons": np.arange(1, 100),
    "model__learning_rate": reciprocal(3e-4, 3e-2),
}

rnd_search_cv = RandomizedSearchCV(keras_reg, param_distribs, n_iter=10, cv=3, verbose=2)

# --- TRAINING ---
print("Mulai training...")
rnd_search_cv.fit(X_train, y_train, epochs=100,
                  validation_data=(X_valid, y_valid),
                  callbacks=[keras.callbacks.EarlyStopping(patience=10)])

print("Best params:", rnd_search_cv.best_params_)

Versi Scikit-Learn saat ini: 1.5.2
Mulai training...
Fitting 3 folds for each of 10 candidates, totalling 30 fits


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


[CV] END model__learning_rate=0.003517136630973405, model__n_hidden=3, model__n_neurons=83; total time= 1.2min


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


[CV] END model__learning_rate=0.003517136630973405, model__n_hidden=3, model__n_neurons=83; total time= 1.1min


...

[CV] END model__learning_rate=0.011455687399508882, model__n_hidden=1, model__n_neurons=76; total time= 1.0min


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


[CV] END model__learning_rate=0.011455687399508882, model__n_hidden=1, model__n_neurons=76; total time= 1.0min


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Best params: {'model__learning_rate': np.float64(0.004656035975849724), 'model__n_hidden': 3, 'model__n_neurons': np.int64(84)}


### Guidelines for Hyperparameters

- Hidden Layers: Start with 1. Deep networks are parameter-efficient due to hierarchical feature learning. Use transfer learning (reuse lower layers of pretrained networks) for complex tasks.
- Neurons: The "stretch pants" approach is often best: pick a model with more layers/neurons than needed and use Early Stopping to prevent overfitting.
- Learning Rate: The most critical hyperparameter. To find the optimal rate, train for a few hundred iterations, exponentially increasing the rate, and plot the loss. Pick a value slightly lower than the point where the loss shoots up.
- Batch Size: Large batch sizes maximize GPU usage but can lead to instability. Small batches (e.g., 32) are often safer, though new techniques (learning rate warmup) allow for larger batches.